# 1. Data Loading

In [ ]:
import pandas as pd

df = pd.read_csv("dirty_cafe_sales.csv")

df.head()

# 2. Initial Exploration

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.nunique()

In [ ]:
df.duplicated().sum()

# 3. Data Type Conversion

In [ ]:
expected_types = {
    "Transaction Date": "datetime64",
    "Item": "object",
    "Quantity": "int",
    "Price Per Unit": "float",
    "Total Spent": "float"
}

In [ ]:
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price Per Unit"] = pd.to_numeric(df["Price Per Unit"], errors="coerce")
df["Total Spent"] = pd.to_numeric(df["Total Spent"], errors="coerce")

In [ ]:
df.dtypes

In [ ]:
df["Quantity"] = df["Quantity"].round().astype("Int64")

In [ ]:
df.dtypes

# 4. Handling Missing Values

In [ ]:
import pandas as pd

df.replace(["ERROR", "UNKNOWN", ""], pd.NA, inplace=True)

In [ ]:
df["Price Per Unit"] = df["Price Per Unit"].fillna(
    df["Total Spent"] / df["Quantity"]
)

In [ ]:
df["Quantity"] = df["Quantity"].fillna(
    df["Total Spent"] / df["Price Per Unit"]
)

In [ ]:
df["Price Per Unit"] = df["Price Per Unit"].fillna(
    df.groupby("Item")["Price Per Unit"].transform("median")
)

In [ ]:
df["Item"] = df["Item"].fillna(
    df.groupby("Price Per Unit")["Item"]
      .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
)

In [ ]:
df["Total Spent"] = df["Quantity"] * df["Price Per Unit"]

In [ ]:
df["Transaction Date"] = df["Transaction Date"].ffill()

In [ ]:
df.isnull().sum()

# 5. Outlier Detection

In [ ]:
pip install scipy

In [ ]:
Q1 = df["Total Spent"].quantile(0.10)
Q3 = df["Total Spent"].quantile(0.90)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df["Total Spent"] < lower) | (df["Total Spent"] > upper)]

outliers

In [ ]:
from scipy import stats

df["zscore"] = stats.zscore(df["Total Spent"], nan_policy="omit")

outliers = df[df["zscore"].abs() > 3]
outliers

In [ ]:
pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt

plt.boxplot(df["Total Spent"].dropna())
plt.title("Outliers in Total Spent")
plt.show()

# 6. Final Summary

In [ ]:
print("Final Summary:")
print("Data shape:", df.shape)
print("Null values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("\nValue counts for key columns:")
print(df["Location"].value_counts(), "\n")
print(df["Item"].value_counts(), "\n")
print(df["Payment Method"].value_counts(), "\n")